# Phase B — 가설 테이블 + 검증 템플릿

**목적**: ai-feat-294 에서 만든 raw eventRows 시각화 도구를 활용하여, 사람/매크로 패턴을 가설 단위로 발굴·검증.

**입력**:
- `eda_gyeom.ipynb` 셀 60 — Phase B 입력 정리 (SMD 순위 + 분포 형태)
- `outputs/294_raw_trajectory/*.png` — 그룹별 trajectory grid
- `visualizer.ipynb` — 그리드 + 단일 trial deep dive helper

**산출**: 본 노트북은 가설 인덱스. 가설 검증은 셀 2 의 템플릿을 복사해서 가설별로 추가.

## 가설 테이블 (B'' — PNG 시각 검증 시드)

초기 4개 (인수 시점) + eda_gyeom 정리에서 도출한 4개 = 총 8개. 발굴 진행 중 자유 추가.

**검증 상태 표기**: `unverified` / `partial` (raw 관찰과 부분 일치) / `verified` / `rejected` (raw 관찰과 불일치) / `trivial` (수집/매크로 알고리즘 산물 의심) / `reframed` (가설 자체를 다시 정의)

| ID | 가설 | 입력 출처 | 예상 매칭 metric (eda_gyeom n=652 기준) | 검증 상태 |
|---|---|---|---|---|
| H1 | **macro 좌표 영역 사용 범위가 좁다** | trajectory PNG (lv2_human vs lv2_macro grid) | `mouse_total_travel_distance_px` 사람 8643 / 매크로 2694 (3×) | **rejected → reframed** (lv2_human 1689 가장 넓음 / lv2_macro 377 좁음 — macro 만 좁음) |
| H1' | **macro 좌표 영역이 사람보다 좁다** (H1 의 reframe) | H1 검증 결과 (반전 발견) | `x_range`, `y_range` (raw 계산) — lv2_human 1689×1075 vs lv2_macro 377×374 | **partial** — lv2 데이터 한정. lv2_collector artifact 가능성 배제 못 함 (Bezier 시작/끝점이 좁은 영역에 고정). 후속: lv3 collector 또는 외부 macro 도구 데이터로 추가 검증 필요 |
| H2 | **macro click 간격이 짧고 일정하다** | speed/dt grid PNG | `inter_click_interval_ms` 사람 7826 / 매크로 1190 (매크로 SMD 0.36 small 이지만 평균치 6.5×) | unverified |
| H3 | **사람 path 가 더 각진/떨림 있다** (직관과 반대 — 매크로 = 직선) | trajectory + speed PNG | `mouse_path_curvature_mean` 0.178 / 0.107 | **dataset-dependent** (lv2 only: macro 0.108 > human 0.063 = reject. Balabit 통합: balabit_human 0.190 > lv2_macro 0.108 = verified). lv2_human(보겸) 이 outlier — 단일 사용자라 평균 사람보다 직선적 |
| H4 | **Balabit 분산 ≫ lv2_human** (게이트 가설) | balabit grid vs lv2_human grid PNG | trial 별 metric cv 비교 — 사람 도메인 일반화 가능 여부 | **partial** (10 metric 중 6 에서 cv_ratio > 1.5) |
| H5 | **macro 평균 속도가 사람보다 빠르다** (직관과 반대 — H1 의 모순?) | speed grid PNG | `mouse_avg_speed_px_per_ms` 사람 0.17 / 매크로 0.23 (매크로 우세) | unverified |
| H6 | **macro path 가 사람보다 더 직선** (global) | trajectory PNG | `mouse_path_straightness_score` 사람 0.066 / 매크로 0.091 (매크로 우세) | **verified** at all groups (lv2_macro 0.091 > lv2_human 0.066, balabit 0.106 mean — median 0.042 outlier 분포) |
| H6' | **H3 + H6 은 모순이 아니라 다른 측면 측정** (LOCAL curvature vs GLOBAL straightness) | H3+H6 정의 비교 + group-wise spearman | spearman ρ ≈ 0 (group 내 두 metric 독립) — 같은 측면 측정 가설 reject | **verified** — 둘 다 macro 우세인 건 좁은 Bezier 가 두 측면 모두에서 우세하다는 매크로 시그니처 |
| H7 | **macro 의 max speed 가 매우 작다** (Bezier 속도 cap 의심) | speed grid PNG (peak 영역) | `mouse_max_speed_px_per_ms` 사람 7.97 / 매크로 1.12 (~7×) | unverified |
| H8 | **pre-click 영역에서 사람의 hover 시간이 더 김** | pre_click grid PNG | `mouse_hover_dwell_time_ms` 사람 845 / 매크로 484 (1.7×) | unverified |

**상호 관계 메모**
- H1 + H7 = macro 가 좁은 영역에서 천천히 움직임 (단, H5 와 모순 — total_distance 짧아도 평균 속도 빠른 이유?)
- H3 + H6 = path 형태 정반대처럼 보였으나 H6' 에서 정리: 두 metric 이 다른 측면 측정 (LOCAL vs GLOBAL), 모순 아님
- H4 = lv2 단일 사용자 vs Balabit 10명, 분산 차이 보이면 사람 도메인 일반화 근거

**Case 분류 (H1 + H4 검증 후)**
- ❌ Case A (lv2_human 좁음 + Balabit 보충): lv2_human 좌표 충분 — 기각
- ❌ Case B (둘 다 좁음): lv2_human 좌표 충분 — 기각
- ⚠ Case C (lv2 충분 + Balabit cv 큼): 좌표만 부분 적용
- 🆕 **Case D**: lv2_human 좌표 충분 / 단일 사용자라 동역학 분산 작음 / Balabit 이 동역학 분산 보충 — **현재 데이터 상태**


## 검증 셀 템플릿 (B''' — 가설별 복사)

각 가설 검증 시 아래 3 셀을 복사해서 사용. `metric_name` / `human_id` / `macro_id` / 가설 ID 만 채워넣으면 됩니다.

**구성**
- ① metric 분포 비교 (사람 vs macro) — boxplot
- ② 사람/macro trial 1개씩 raw plot (4 plot 2 row) — plots.py 활용
- ③ "metric 값이 raw 관찰과 일치하는가" 1줄 코멘트 마크다운

In [ ]:
# 템플릿 — 모든 검증 셀의 공통 import (1회만 실행)
%matplotlib inline
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from trial_loader import (
    DATA_DIR, GROUPS,
    list_trials_by_group, load_trial,
)
from plots import (
    plot_trajectory, plot_speed_over_time,
    plot_dt_distribution, plot_pre_click_paths,
)

In [ ]:
# === H? 검증 — Step ① metric 분포 비교 ===
# TODO: metric_name 채우기
metric_name = "TODO_metric_name"  # 예: "mouse_total_travel_distance_px"

rows = []
for path in sorted(DATA_DIR.glob("trial_*.json")):
    d = json.loads(path.read_text(encoding="utf-8"))
    label = d.get("label")
    if label not in ("human", "macro"):
        continue
    val = (d.get("metrics") or {}).get(metric_name)
    if val is not None:
        rows.append({"label": label, metric_name: val})
metric_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=metric_df, x="label", y=metric_name, hue="label",
            order=["human", "macro"], legend=False, dodge=False, ax=ax)
sns.stripplot(data=metric_df, x="label", y=metric_name,
              order=["human", "macro"], color="black", alpha=0.25, size=2, jitter=0.2, ax=ax)
ax.set_title(f"{metric_name}  (n_human={metric_df.label.eq('human').sum()}, n_macro={metric_df.label.eq('macro').sum()})")
plt.tight_layout()
plt.show()

print(metric_df.groupby("label")[metric_name].describe()[["count", "mean", "std", "min", "50%", "max"]])

In [ ]:
# === H? 검증 — Step ② raw plot (사람/매크로 1 trial 씩) ===
# TODO: trial_id 선택 (분포 양 끝 또는 중앙값 근처)
human_id = list_trials_by_group("lv2_human")[0]["trial_id"]
macro_id = list_trials_by_group("lv2_macro")[0]["trial_id"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for row, (label, tid) in enumerate([("human", human_id), ("macro", macro_id)]):
    trial = load_trial(tid)
    plot_trajectory(trial,      ax=axes[row, 0])
    plot_speed_over_time(trial, ax=axes[row, 1])
    plot_dt_distribution(trial, ax=axes[row, 2])
    plot_pre_click_paths(trial, ax=axes[row, 3])
    axes[row, 0].set_ylabel(f"{label}\n{axes[row, 0].get_ylabel()}", fontsize=9)
fig.suptitle(f"H? raw eventRows  (human={human_id}, macro={macro_id})", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

### H? 관찰 결과 (Step ③)

**Metric 값**: TODO  
**Raw 관찰**: TODO  
**일치 여부**: ☐ 일치 / ☐ 부분 일치 / ☐ 불일치 / ☐ 트리비얼 의심  
**메모 (1 줄)**: TODO

## 검증된 가설 (아래에 누적)

가설 검증 시 위 템플릿을 복사하여 이 헤더 아래에 추가, 가설 테이블의 "검증 상태" 컬럼도 갱신.

---

## H1 검증 — 좌표 범위 분포

**가설**: macro 좌표 영역이 좁다 (인수 시드는 lv2_human 도 좁아 artifact 의심).
**검증**: 그룹별 trial 의 (x_max - x_min), (y_max - y_min) 분포 비교.

In [ ]:
# === H1 검증: 좌표 범위 (x_max - x_min, y_max - y_min) 분포 ===
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from trial_loader import GROUPS, list_trials_by_group, load_trial, event_rows_to_df

rows = []
for group in GROUPS:
    for meta in list_trials_by_group(group):
        trial = load_trial(meta["trial_id"])
        df = event_rows_to_df(trial)
        mv = df[df["is_move"]].dropna(subset=["x", "y"])
        if len(mv) < 5:
            continue
        rows.append({
            "group": group,
            "trial_id": meta["trial_id"],
            "x_range": float(mv["x"].max() - mv["x"].min()),
            "y_range": float(mv["y"].max() - mv["y"].min()),
            "x_min": float(mv["x"].min()), "x_max": float(mv["x"].max()),
            "y_min": float(mv["y"].min()), "y_max": float(mv["y"].max()),
        })

range_df = pd.DataFrame(rows)
print("n trials:", range_df.groupby("group").size().to_dict())
print()
for col in ["x_range", "y_range"]:
    print(f"--- {col} ---")
    print(range_df.groupby("group")[col].describe()[["count","mean","std","min","25%","50%","75%","max"]].round(1))
    print()
print("--- 평균 좌표 영역 ---")
for g in ["lv2_human", "lv2_macro", "balabit"]:
    sub = range_df[range_df["group"] == g]
    if not sub.empty:
        print(f"  {g:12s}: x [{sub['x_min'].mean():.0f} ~ {sub['x_max'].mean():.0f}], "
              f"y [{sub['y_min'].mean():.0f} ~ {sub['y_max'].mean():.0f}]")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
order = ["lv2_human", "lv2_macro", "balabit"]
sns.boxplot(data=range_df, x="group", y="x_range", order=order, ax=axes[0])
sns.stripplot(data=range_df, x="group", y="x_range", order=order, color="black", alpha=0.3, size=2, jitter=0.2, ax=axes[0])
axes[0].set_title("x_range distribution")
sns.boxplot(data=range_df, x="group", y="y_range", order=order, ax=axes[1])
sns.stripplot(data=range_df, x="group", y="y_range", order=order, color="black", alpha=0.3, size=2, jitter=0.2, ax=axes[1])
axes[1].set_title("y_range distribution")
plt.tight_layout()
plt.show()

### H1 결과 (Step ③)

**판정**: ☑ **REJECTED → REFRAMED** (원 가설은 reject, 반전 발견 등록)

| group | x_range mean | y_range mean | 평균 좌표 영역 |
|---|---:|---:|---|
| **lv2_human** | **1689** | **1075** | (1019~2708, 563~1638) ← **가장 넓음** |
| balabit | 923 | 659 | (112~1035, 91~750) |
| **lv2_macro** | **377** | **374** | (824~1200, 420~794) ← **좁음 정사각형** |

**원래 가설 (lv2_human 좁음 = lv2 artifact 의심) 기각**:
- lv2_human 의 x_range mean 1689 / y_range mean 1075 으로 3 그룹 중 가장 넓음
- max x 가 2708 까지 — 듀얼 모니터 또는 고해상도 디스플레이 흔적
- → **원래 H1 (lv2_human 좁음 = artifact) REJECTED**

**신규 H1' 발견 (lv2_macro 만 좁다)**:
- lv2_macro x 377 / y 374 (거의 정사각형) — Bezier 시작/끝점이 좁은 영역 안에서만 움직이는 lv2_collector 시그니처
- Balabit 좌표가 lv2_human 보다 좁음은 RDP 환경 해상도 (~1024×768) 때문 — artifact 아님
- → **신규 H1' 가설 등록** (가설 테이블 참조). 단 lv2_collector 의 target 좌표 고정 가능성 (Bezier 가 *어떤 좌표를 어디에서 어디로* 그리는가) 도 의심해야 함 — lv3 collector / playwright macro 등 타 매크로로 확장 시 무력화 가능

**lv2_human 단일 사용자 overfit 위험 (H4 와 연결)**:
- lv2_human 은 보겸 1인 데이터 (n=51) — 좌표 영역은 넓으나 동역학 분포는 단일 사용자 한계
- H4 분석에서 보겸 동역학(avg_speed/max_speed/acceleration/jerk) 이 Balabit 10명 평균과 **4~19배 차이**
- → 295 1차 5 feature 모델 recall 0.97 의 일반화 의심 정당화. lv2 단독 학습 모델은 보겸 마우스 패턴에 overfit 위험

**Raw 관찰**: macro trajectory PNG 와 정합. lv2_macro 의 좌표 영역이 시각적으로 좁고 정사각형 안에 모여있는 것 확인됨.

**일치 여부**: ☐ 일치 / ☐ 부분 일치 / ☑ 불일치 (가설 reframe 필요)
**메모**: H1 reject + H1' 등록. 단일 사용자 overfit 위험은 별도 게이트 (H4) 에서 추적.

## H4 검증 — Balabit 분산 ≫ lv2_human?

**가설**: lv2_human (n=51, 단일 사용자 보겸) 의 metric 분산이 Balabit (n=500, 10 사용자) 보다 작다 (게이트 가설).
**검증**: 16 USABLE metric 의 cv = std / |mean| 비교.

In [ ]:
# === H4 검증: 16 USABLE metric cv = std / |mean| 비교 (사람 도메인 분산) ===
import json, numpy as np, pandas as pd
from trial_loader import DATA_DIR

USABLE_16 = [
    "durationMs", "clickCount", "eventCount",
    "time_to_first_click_ms", "inter_click_interval_ms", "pre_click_mousemove_count",
    "mouse_total_travel_distance_px", "mouse_avg_speed_px_per_ms", "mouse_max_speed_px_per_ms",
    "mouse_speed_change_mean", "mouse_acceleration_mean", "mouse_jerk_mean",
    "mouse_path_straightness_score", "mouse_path_curvature_mean", "mouse_direction_change_count",
    "mouse_hover_dwell_time_ms", "mouse_stop_segment_count", "mousemove_event_rate",
]

trial_rows = []
for path in sorted(DATA_DIR.glob("trial_*.json")):
    d = json.loads(path.read_text(encoding="utf-8"))
    if d.get("label") != "human":
        continue
    tid = int(d.get("trialId"))
    if 900001 <= tid <= 909999:
        sub = "lv2_human"
    elif 910001 <= tid <= 910500:
        sub = "balabit"
    else:
        continue
    row = {"sub": sub, "trial_id": tid}
    metrics = d.get("metrics") or {}
    for m in USABLE_16:
        row[m] = metrics.get(m)
    trial_rows.append(row)

mdf = pd.DataFrame(trial_rows)
print("n trials:", mdf.groupby("sub").size().to_dict())
print()

def cv(s):
    if len(s) < 5 or abs(s.mean()) < 1e-12:
        return np.nan
    return s.std() / abs(s.mean())

cv_results = []
for m in USABLE_16:
    lv2 = pd.to_numeric(mdf.loc[mdf["sub"] == "lv2_human", m], errors="coerce").dropna()
    bal = pd.to_numeric(mdf.loc[mdf["sub"] == "balabit",   m], errors="coerce").dropna()
    cv_results.append({
        "metric": m,
        "n_lv2": len(lv2), "cv_lv2": cv(lv2),
        "n_bal": len(bal), "cv_bal": cv(bal),
        "median_lv2": lv2.median() if len(lv2) else np.nan,
        "median_bal": bal.median() if len(bal) else np.nan,
    })

cdf = pd.DataFrame(cv_results)
cdf["cv_ratio"] = cdf["cv_bal"] / cdf["cv_lv2"]
cdf_filled = cdf.dropna(subset=["cv_lv2", "cv_bal"]).sort_values("cv_ratio", ascending=False)

print("--- cv 비교 (양쪽 모두 채워진 metric) ---")
print(cdf_filled[["metric","n_lv2","cv_lv2","n_bal","cv_bal","cv_ratio","median_lv2","median_bal"]].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print()
print(f"Balabit cv >> lv2_human cv (>1.5): {(cdf_filled['cv_ratio'] > 1.5).sum()} / {len(cdf_filled)}")
print(f"비슷 (0.7 ~ 1.5):                   {((cdf_filled['cv_ratio'] >= 0.7) & (cdf_filled['cv_ratio'] <= 1.5)).sum()} / {len(cdf_filled)}")
print(f"lv2_human cv >> Balabit cv (<0.7):  {(cdf_filled['cv_ratio'] < 0.7).sum()} / {len(cdf_filled)}")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11, 5))
y = list(range(len(cdf_filled)))
ax.barh(y, cdf_filled["cv_ratio"], color="steelblue")
ax.axvline(1.0, color="gray", linestyle="--", alpha=0.5)
ax.axvline(1.5, color="red", linestyle="--", alpha=0.5, label="cv_ratio = 1.5")
ax.set_yticks(y)
ax.set_yticklabels(cdf_filled["metric"], fontsize=9)
ax.set_xlabel("cv_ratio = cv_balabit / cv_lv2_human")
ax.set_title("H4: Balabit cv vs lv2_human cv (>1 = Balabit 분산 큼)")
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### H4 결과 (Step ③)

**판정**: ☑ **PARTIAL VERIFIED** — 10 metric 중 6 에서 Balabit cv 가 lv2_human cv 보다 1.5배 이상 큼.

| ratio | metric | cv_lv2 / cv_bal |
|---:|---|---|
| 18.7× | inter_click_interval_ms | 0.17 / 3.22 |
|  2.8× | mouse_path_straightness_score | 0.61 / 1.71 |
|  2.4× | mouse_total_travel_distance_px | 0.28 / 0.69 |
|  2.4× | mouse_avg_speed_px_per_ms | 0.29 / 0.69 |
|  1.9× | mouse_speed_change_mean | 0.34 / 0.66 |
|  1.9× | mouse_max_speed_px_per_ms | 0.88 / 1.66 |
|  1.4× | mouse_direction_change_count | 0.27 / 0.40 |
|  1.4× | mouse_acceleration_mean | 0.79 / 1.08 |
|  1.2× | mouse_path_curvature_mean | 0.24 / 0.28 |
| 0.86× | mouse_jerk_mean | 1.91 / 1.63 |

**강한 추가 발견 — median 동역학 격차**:

| metric | lv2_human (보겸) | Balabit (10명) | 비율 |
|---|---:|---:|---:|
| mouse_avg_speed | 0.51 | 0.12 | 4× |
| mouse_max_speed | 15.0 | 4.3 | 3.5× |
| mouse_acceleration | 0.072 | 0.0037 | **19×** |
| mouse_jerk_mean | 0.009 | ~0 | ≫ |
| mouse_path_curvature | 0.064 | 0.186 | **0.34×** (역방향) |
| inter_click_interval | 2265 | 3640 | 0.62× |

→ **사람 마다 동역학이 매우 다름**. lv2_human (보겸 단독) 은 Balabit 10명 평균 사람보다 **4~19배 빠르고 격렬**, 곡률은 **3분의 1 수준**.
→ **H3 (사람 path 가 매크로보다 떨림/각짐) 의심 등록**: lv2_human 곡률 0.064 < lv2_macro 0.107 — 보겸 마우스가 lv2 macro 보다 직선적. Balabit (0.186) 으로 평균 사람을 보면 가설 성립.

**판정 종합**:
- ☑ **PARTIAL VERIFIED** (10개 중 6개 metric 에서 Balabit cv ≥ 1.5× lv2_human)
- ☑ **Case D 채택** — lv2_human 좌표 측면 충분, 동역학 측면 Balabit 보완
- ☐ **ADR amendment 트리거 X** — metric 으로 잡힌 가설 = 100% 가 아직 검증 완료된 상태가 아니므로 도메인 결정 문서 갱신은 보류

**Case 분류 갱신 (H1 + H4 종합)**
- ❌ Case A / Case B: lv2_human 좌표 충분으로 기각
- ⚠ Case C (lv2 충분 + Balabit cv 큼): 좌표 측면만 일치
- 🆕 **Case D**: lv2_human 좌표 충분 / 단일 사용자 동역학 분산 작음 / Balabit 동역학 분산 보충 — **현재 데이터 상태**

**게이트 통과 여부**: ☑ **partial pass**
- Balabit 으로 사람 도메인 동역학 분산 보충 가능
- 단 Balabit 채움 metric 14개로 한정 — `mouse_hover_dwell_time_ms`, `mouse_stop_segment_count`, `mousemove_event_rate` 는 Balabit 미수집
- **다음 가설 진입 가능** (특히 H7 = max_speed Bezier cap, H3 = path 떨림은 Balabit 풀에서 재검증)

**Raw 관찰**: trajectory grid PNG 에서 Balabit 5×4 grid 의 다양성 vs lv2_human 4×4 grid (보겸 1인) 의 단조로움 — 시각적 일치.

**메모**: 단순 cv 비교가 H4 검증 충분조건은 아님 (분산이 크다 ≠ 다양한 사람을 잘 대표한다). user_id 별 within-cluster vs between-cluster 분산 비교까지 갈 필요는 있으나 Phase B 다음 step 으로 미룸.

## H3 + H6 검증 — curvature vs straightness 모순 정리

**가설**:
- H3: 사람 path 가 더 각진/떨림 (curvature_mean 사람 우세)
- H6: macro path 가 더 직선 (straightness_score 매크로 우세)
- → eda_gyeom 셀 14 에서 lv2 데이터 한정으로는 **둘 다 macro 가 큼** (curvature 0.108 > 0.063, straightness 0.091 > 0.066) — 모순처럼 보임.

**목적**: metric 정의 차이인지 진짜 모순인지 판정. curvature_mean 은 학습 핵심 metric (SMD 1.51) 이라 신뢰도 확정 필요.

### Step 1 — Metric 정의 확인

소스 코드 직접 확인 (`balabit_to_trial.py:131-135`, `balabit_metrics_augment.py:121-126`).

| Metric | 정의 (정본: collector core.js) | 측정 측면 |
|---|---|---|
| `mouse_path_curvature_mean` | `mean(direction_change_angle / segment_distance)` over segments | **LOCAL** — 단위거리당 각도 변화. path 가 얼마나 자주/크게 꺾이는지 |
| `mouse_path_straightness_score` | `distance(first, last) / total_travel` | **GLOBAL** — 시작-끝 직선거리 / 실제 path 총거리. 1=완전직선, 0=제자리 |

**핵심**: 두 metric 은 **완전히 다른 측면**을 측정.

- LOCAL curvature: segment 간 각도 변화의 단위거리 평균. 짧은 segment 에서 작은 각도 변화도 크게 잡힘.
- GLOBAL straightness: 시작점에서 끝점으로 얼마나 효율적으로 갔는지. 왕복하면 작아짐.

**같은 측면 측정 가설 reject 의 근거** (group-wise spearman 상관):

| group | spearman ρ (curvature vs straightness) | p-value | 해석 |
|---|---:|---:|---|
| lv2_human | -0.002 | 0.99 | 거의 독립 |
| lv2_macro | -0.112 | 0.27 | 약한 음의 상관 (유의 X) |
| balabit | +0.120 | 0.007 | 약한 양의 상관 (유의 O 단 약함) |

→ group 내에서 두 metric 이 거의 독립. **다른 측면 측정 확정**.

In [ ]:
# === Step 2: 두 metric 분포 비교 (3 그룹) ===
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from trial_loader import DATA_DIR

rows = []
for path in sorted(DATA_DIR.glob("trial_*.json")):
    d = json.loads(path.read_text(encoding="utf-8"))
    label = d.get("label")
    tid = int(d.get("trialId"))
    if label not in ("human", "macro"):
        continue
    if 900001 <= tid <= 909999:
        group = "lv2_human" if label == "human" else "lv2_macro"
    elif 910001 <= tid <= 910500:
        group = "balabit"
    else:
        continue
    metrics = d.get("metrics") or {}
    rows.append({
        "group": group, "trial_id": tid,
        "curvature": metrics.get("mouse_path_curvature_mean"),
        "straightness": metrics.get("mouse_path_straightness_score"),
    })
mdf = pd.DataFrame(rows)

print("=== 통계 요약 ===")
order = ["lv2_human", "lv2_macro", "balabit"]
for col in ["curvature", "straightness"]:
    print(f"\n[{col}]")
    print(mdf.groupby("group")[col].describe()[["count","mean","std","min","25%","50%","75%","max"]].reindex(order).round(4))

print("\n=== group 내 spearman 상관 (curvature vs straightness) ===")
from scipy.stats import spearmanr
for g in order:
    sub = mdf[mdf["group"] == g][["curvature", "straightness"]].dropna()
    if len(sub) >= 5:
        r, p = spearmanr(sub["curvature"], sub["straightness"])
        print(f"  {g:12s}: rho={r:+.3f}  p={p:.4f}  n={len(sub)}")

# violin + boxplot
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for col_i, col in enumerate(["curvature", "straightness"]):
    sns.violinplot(data=mdf, x="group", y=col, order=order, ax=axes[col_i, 0], cut=0, inner="quartile")
    axes[col_i, 0].set_title(f"{col} — violin (full distribution)")
    sns.boxplot(data=mdf, x="group", y=col, order=order, ax=axes[col_i, 1], showfliers=True)
    sns.stripplot(data=mdf, x="group", y=col, order=order, color="black", alpha=0.2, size=2, jitter=0.2, ax=axes[col_i, 1])
    axes[col_i, 1].set_title(f"{col} — boxplot")
plt.tight_layout()
plt.show()

# scatter — curvature vs straightness
fig, ax = plt.subplots(figsize=(8, 6))
for g, color in zip(order, ["tab:blue", "tab:red", "tab:green"]):
    sub = mdf[mdf["group"] == g][["curvature", "straightness"]].dropna()
    ax.scatter(sub["curvature"], sub["straightness"], s=14, alpha=0.5, label=f"{g} (n={len(sub)})", color=color)
ax.set_xlabel("curvature_mean (LOCAL)")
ax.set_ylabel("straightness_score (GLOBAL)")
ax.set_title("curvature vs straightness — group 별 산점도 (group 내 ρ ≈ 0)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# === Step 3: raw 샘플 검증 — 각 그룹 median curvature trial 1개씩 trajectory 비교 ===
from trial_loader import load_trial
from plots import plot_trajectory

# Step 2 의 mdf 사용. curvature median 에 가장 가까운 trial 선택
sample_ids = {}
for g in ["lv2_human", "lv2_macro", "balabit"]:
    sub = mdf[mdf["group"] == g][["trial_id", "curvature", "straightness"]].dropna().copy()
    if sub.empty:
        continue
    med = sub["curvature"].median()
    sub["d"] = (sub["curvature"] - med).abs()
    pick = sub.sort_values("d").iloc[0]
    sample_ids[g] = (int(pick["trial_id"]), float(pick["curvature"]), float(pick["straightness"]))
    print(f"  {g:12s}: trial_{int(pick['trial_id'])}  curv={pick['curvature']:.4f}  straight={pick['straightness']:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (g, (tid, curv, straight)) in zip(axes, sample_ids.items()):
    trial = load_trial(tid)
    plot_trajectory(trial, ax=ax)
    ax.set_title(f"{g}\ntrial_{tid}\ncurvature={curv:.4f}, straightness={straight:.4f}", fontsize=9)
fig.suptitle("Median curvature trial 의 trajectory — metric 값 vs raw 형태 매칭", fontsize=11)
plt.tight_layout()
plt.show()

### Step 4 — 판정

**모순이 아니다**:

| 측면 | 사람 (lv2_human, 보겸) | 매크로 (lv2_macro) | Balabit human (10명 평균) |
|---|---|---|---|
| 좌표 영역 | 매우 넓음 (1689×1075) | 좁음 (377×374) | 중간 (923×659) |
| total_travel | 매우 큼 (8643) | 작음 (2694) | 큼 |
| direct/total (straightness) | 매우 작음 (0.066) — 큰 영역 배회 | 약간 큼 (0.091) — 좁은 영역의 시작-끝 | 가장 작음 median (0.042) — 떨림 큼 |
| curvature (단위거리당) | 매우 작음 (0.063) — 긴 segment 적은 꺾임 | 적당 (0.108) — Bezier 부드러운 곡선 | 큼 (0.190) — 잦은 떨림 |

**해석**:
1. **사람(보겸)**: 큰 영역에서 직선적으로 이동 (긴 segment, 작은 곡률) + 왕복/배회로 total_travel ≫ direct (straightness 작음)
2. **매크로(lv2_collector)**: 좁은 영역에서 Bezier 곡선 — 짧은 segment 마다 부드러운 각도 변화 (단위거리당 곡률 큼) + 시작-끝이 좁은 영역 안의 다른 click 지점 (straightness 약간 큼)
3. **Balabit (평균 사람)**: RDP 환경의 자연스러운 떨림 — 잦은 작은 각도 변화 (curvature 큼) + 떨림 누적으로 total ≫ direct (straightness 작음)

**두 metric 모두 macro 가 큰 게 lv2 한정 결과로 나타나는 이유**:
- 매크로의 *좁은 Bezier* 패턴이 두 측면 (단위거리당 곡률, 시작-끝 비율) 모두에서 lv2_human (큰 영역 배회) 보다 큰 값을 만들어냄
- group 내 spearman ρ ≈ 0 → 두 metric 이 같은 측면을 잡는 게 아니라 **다른 두 측면이 우연히 같은 방향으로 macro 우세**
- → 모순 아님. 두 metric 의 정의가 다르다는 점이 핵심.

**판정**:
- ☑ **H3 (사람 path 더 각진/떨림): dataset-dependent**
  - lv2 only: macro curvature (0.108) > lv2_human (0.063) → **REJECTED**
  - Balabit 통합: balabit_human (0.190) > lv2_macro (0.108) → **VERIFIED**
  - lv2_human (보겸) 단일 사용자가 outlier: 매크로보다 직선적인 마우스 사용자
- ☑ **H6 (macro 가 더 직선): VERIFIED at lv2 level**
  - lv2_macro (0.091) > lv2_human (0.066), balabit median (0.042) 은 outlier 분포 (mean 0.106)
- 🆕 **H6' (모순 아님): VERIFIED**
  - 두 metric 이 다른 측면 측정 (LOCAL curvature, GLOBAL straightness)
  - group 내 ρ ≈ 0 으로 독립 확정
  - lv2 한정으로 둘 다 macro 우세인 건 매크로의 좁은 Bezier 가 두 측면 모두에서 사람의 큰 영역 배회보다 큰 값을 내기 때문

**Raw 관찰**: 위 sample trial 의 trajectory PNG 가 metric 값과 정합:
- lv2_human (curv 0.064, straight 0.063): 큰 영역 배회 path 시각 확인
- lv2_macro (curv 0.109, straight 0.047): 좁은 정사각형 안 부드러운 곡선
- balabit (curv 0.186, straight 0.316): 떨림 + 짧은 path

### Step 5 — `mouse_path_curvature_mean` 학습 신뢰도

**SMD 1.51 (very_large), 295 1차 5 feature 학습 핵심 metric** 으로서 다음 신뢰도 평가:

| 항목 | 평가 |
|---|---|
| 정의 명확성 | ☑ — `direction_change_angle / segment_distance` 의 평균. 정본은 collector core.js (`deriveMetrics`) 에 있으며 balabit_to_trial 이 vendored. |
| 도메인 의미 | ☑ — 단위거리당 각도 변화 = path 의 떨림/굽음 정도. 사람의 마우스 jitter 와 macro 의 부드러운 Bezier 차이를 잡는 의미 있는 측면. |
| 사용자 의존성 | ⚠ — lv2_human (보겸 단일) 0.064 vs Balabit 평균 0.190 = **3×**. 사람마다 mouse 사용 스타일이 다름. |
| **lv2 단독 학습 시 reverse 위험** | ⚠⚠ — lv2_macro (0.108) > lv2_human (0.063) → 모델이 *macro = 곡선 큼* 으로 학습. lv3 collector 또는 다른 사용자 추가 시 무력화. |
| **Balabit 풀 포함 시** | ☑ — balabit_human (0.190) > macro (0.108) 으로 정합. 평균 사람 분포에서 사람 우세. |

**결론**:
- ☑ **Balabit 풀과 함께 학습 시 신뢰도 OK**. 평균 사람 (10명) 의 떨림이 매크로 Bezier 보다 큰 곡률을 만들어 정합한 시그니처.
- ⚠ **lv2 단독 학습은 reverse 위험**. 보겸 마우스가 매크로보다 직선적이라 학습된 결정 경계가 *반대 방향* 이 됨.
- 🛡 **lv3 collector 단계 진입 전 권장**:
  1. lv2 단독 모델은 retired 또는 baseline 으로만 사용
  2. 학습 입력 = lv2 + Balabit (사람측) + lv2_collector (매크로) 풀
  3. user_id 별 within/between cluster 분산 분리 (Phase B 후속)
  4. 가능하면 다중 사용자 lv3 collector 데이터 추가 후 재학습

**ADR amendment 트리거 X** — metric 정의 자체는 변경 불요 (정의 명확). 학습 데이터 풀과 사용자 다양성 확보가 우선.